In [13]:
import pandas as pd
import numpy as np
from collections import Counter

In [14]:
#root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
root = "Y:/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
outputFolder = root + "DataFrames" + "/"

In [15]:
def lineages(x, links_dataframe):
    # Parse lineage relationships between splitting events
    source_id = x.Source_ID # each row's source spot ID
    # add source id to list if the row is a splitting event
    if x.Splitting_event:
        lineage = [str(source_id)]
    else:
        lineage = []
    while True:
        target_id = links_dataframe.loc[links_dataframe['Target_ID'] == source_id,:] # find row containing corresponding target ID
        if target_id.empty:
            break
        if target_id.Splitting_event.values[0]:
            lineage.append(str(target_id.Source_ID.values[0]))
        source_id = target_id.Source_ID.values[0]
    if lineage:    
        return ".".join(reversed(lineage)) # insert . between every element in the list, in reverse order
    else:
        return None

def get_mother(x):
    lineage_list = x.split(".")
    if len(lineage_list) == 1:
        return None
    else:
        return lineage_list[-2] 

def get_grandmother(x):
    lineage_list = x.split(".")
    if len(lineage_list) < 3:
        return None
    else:
        return lineage_list[-3]     
    
def get_sister(x, dataframe):
    if x.Mother_ID is None:
        return None
    else:
        sister_df = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Mother_ID == x.Mother_ID) & 
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID != x.Source_ID), :
        ]
        if sister_df.shape[0] > 0:
            lineage_sister = sister_df.iloc[0]['Lineage']
            sister = lineage_sister.split(".")[-1]
            return sister
        else:
            pass

def get_daughters(x, dataframe):
    daughters_df = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Track_ID == x.Track_ID) &
            (dataframe.Mother_ID == x.Source_ID) & 
            (dataframe.Position == x.Position), :
    ]
    daughters_list = daughters_df.Source_ID.unique()
    return daughters_list

def get_mother_n_of_daughters(x, dataframe):

    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID) &
            (dataframe.Position == x.Position)
        ]
        
        n_daughters = mother_row.Number_of_daughters_in_mitosis.item()
        return n_daughters

def seensister(x, already_seen_list):
    if x.Sister_ID is None:
            return None
    else:
        if x.Mother_ID in already_seen_list:
            return True
        else:
            already_seen_list.append(x.Mother_ID)
            return False

def get_cousin(x, dataframe):
    if x.Grandmother_ID is None:
        return None
    else:
        cousins_df = dataframe.loc[
            (dataframe.Grandmother_ID == x.Grandmother_ID) & 
            (dataframe.Source_ID != x.Source_ID) &
            (dataframe.Mother_ID != x.Mother_ID), :
        ]
        cousins = []
        if cousins_df.shape[0] == 2:   
            cousinA = cousins_df.iloc[0, 1].item()
            cousinB = cousins_df.iloc[1, 1].item()
            return cousinA, cousinB
        elif cousins_df.shape[0] == 1:
            cousinA = cousins_df.iloc[0, 1].item()
            return cousinA
        else:
            return None

def get_random(x, dataframe):
    random_df = dataframe.loc[
            (dataframe.Dataset == x.Position) &
            (dataframe.Position == x.Position) &
            (dataframe.Track_ID != x.Track_ID) & # Don't pair within same lineage
            (dataframe.Source_ID != x.Sister_ID) & 
            (dataframe.Source_ID != x.Mother_ID)  
           # (dataframe.Source_ID != x.Grandmother_ID) & 
           # (dataframe.Source_ID != x.Cousin_ID) & 
           # (dataframe.Source_ID != x.Aunt_ID), :
        ]
    
    random_ID = random_df["Source_ID"].sample(1)
    return random_ID.item()

def seengranny(x, granny_seen_list):
    if x.Grandmother_ID is None:
        return None
    else:
        if x.Grandmother_ID not in granny_seen_list:
            granny_seen_list.append(x.Grandmother_ID)
            return False
        else:
            return True

def get_cell_cycle(x, dataframe):
    daughter_time = x.Frame
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        if mother_row["Frame"].shape[0] == 1:   
            mother_time = mother_row["Frame"].item()
            cell_cycle = (int(daughter_time) - int(mother_time)) * 7 # Frame Interval is still hard-coded
            return cell_cycle
        else:
            return None


          
def tracking_dataframes(edges_dir, spots_dir, dataset, position, condition, outdir, max_frame = 823):
    links = pd.read_csv(
        edges_dir,
        usecols = ['TRACK_ID', 'SPOT_SOURCE_ID', 'SPOT_TARGET_ID']
    )
    links.rename(columns = {
        "TRACK_ID": "Track_ID", 
        "SPOT_SOURCE_ID": "Source_ID", 
        "SPOT_TARGET_ID": "Target_ID"
    }, inplace = True)
    
    spots = pd.read_csv(
        spots_dir,
        usecols = ['ID', 'TRACK_ID', 'POSITION_X', 'POSITION_Y', 'FRAME']
    )
    spots.rename(columns = {
        "TRACK_ID": "Track_ID", 
        "POSITION_X": "Track_Coordinate_X", 
        "POSITION_Y": "Track_Coordinate_Y", 
        "FRAME": "Frame"
    }, inplace = True)

    # Crop spots by frame
    spots["Frame"] = pd.to_numeric(spots["Frame"], errors = "coerce")
    spots = spots[spots["Frame"] < max_frame]

    # Crop links based on remaining spots
    valid_ids = set(spots["ID"])

    links = links[
        links["Source_ID"].isin(valid_ids) &
        links["Target_ID"].isin(valid_ids)
    ]
    
    # Find spots that don't have a target and are not at the end of the movie

    # Find all end-point IDs = Target_IDs that never appear as Source_IDs (per Track_ID)
    end_points = (
        links
        .loc[~links['Target_ID'].isin(links['Source_ID'])]
        .groupby('Track_ID')['Target_ID']
        .unique()
        .reset_index(name = 'EndPoint_ID')
    )

    spots_with_endpoints = spots.merge(
        end_points.explode('EndPoint_ID'),
        left_on = ['Track_ID', 'ID'],
        right_on = ['Track_ID', 'EndPoint_ID'],
        how = 'inner'
    )

    last_frame = spots_with_endpoints.Frame.max()
    spots_with_endpoints = spots_with_endpoints[spots_with_endpoints["Frame"] < last_frame]
    
    # Find all downstream spots of each endpoint
    
    from collections import defaultdict
    
    downstream_mapping = []  # will hold {Track_ID, EndPoint_ID, Downstream_IDs}
    
    for track_id, group in links.groupby("Track_ID"):
        # Build adjacency list (Source -> Target)
        adjacency = defaultdict(list)
        for _, row in group.iterrows():
            adjacency[row["Source_ID"]].append(row["Target_ID"])
    
        # Build reverse mapping (Target -> Source) to go upstream
        reverse_adj = defaultdict(list)
        for src, targets in adjacency.items():
            for tgt in targets:
                reverse_adj[tgt].append(src)
    
        # Find endpoints for this track
        endpoints = group.loc[~group["Target_ID"].isin(group["Source_ID"]), "Target_ID"].unique()
    
        for endpoint in endpoints:
            visited = set()
            stack = [endpoint]
    
            while stack:
                current = stack.pop()
                if current in visited:
                    continue
                visited.add(current)
                stack.extend(reverse_adj[current])  # move upstream
    
            downstream_mapping.append({
                "Track_ID": track_id,
                "EndPoint_ID": endpoint,
                "Downstream_IDs": list(visited)
            })

    # Convert to DataFrame
    downstream_df = pd.DataFrame(downstream_mapping)

    # Merge with filtered endpoints

    downstream_df = spots_with_endpoints.merge(downstream_df) 

    destination_downstream = outdir + "_downstream.csv"
    downstream_df.to_csv(destination_downstream)
    print("Successfully exported endpoint dataframe to " + destination_downstream)

    # Generate a list of spot_ids that correspond to a splitting event
    # (Per definition, a splitting event is labelled twice as "source")
    source_ids = list(links["Source_ID"])
    source_id_counts = Counter(source_ids)
    splitting_event_ids = [id for id in source_id_counts if source_id_counts[id] > 1]
    
    # Add Boolean to Spots and Links dataframes
    # that indicate whether the spot or links belongs to a 
    # splitting event

    spots["Splitting_event"] = spots["ID"].apply(lambda x:\
                                                 False if x not in splitting_event_ids\
                                                 else True)

    links["Splitting_event"] = links["Source_ID"].apply(lambda x:\
                                                 False if x not in splitting_event_ids\
                                                 else True)

    
    ###### TODO: DOUBLE CHECK THIS WORKS #####
    #df = df[df["Frame"].astype(int) < max_frame]
    
    links['Lineage'] = links.apply(lineages, args = (links,), axis = 1)
    print("Successfully annotated lineage information.")

    links = links[links["Splitting_event"] == True]
    spots = spots[spots["Splitting_event"] == True]
    spots.rename(columns = {"ID": "Source_ID"}, inplace = True)
    
    
    df = pd.merge(links, spots, how = "outer", on = ["Source_ID", "Track_ID", "Splitting_event"])

    
    
    df.drop(["Target_ID"], axis = 1, inplace = True)
    df.drop_duplicates(subset = ['Source_ID'], inplace = True)




    downstream_df.drop(["Track_Coordinate_X"], axis = 1, inplace = True)
    downstream_df.drop(["Track_Coordinate_Y"], axis = 1, inplace = True)
    downstream_df.drop(["Frame"], axis = 1, inplace = True)
    # First, explode downstream_df so each Downstream_ID is its own row
    downstream_exploded = downstream_df.explode("Downstream_IDs")
    
    # Now downstream_exploded looks like:
    # Track_ID | EndPoint_ID | Downstream_IDs
    
    # Merge df with downstream_exploded on Source_ID == Downstream_IDs
    df_with_endpoints = df.merge(
        downstream_exploded,
        left_on = "Source_ID",
        right_on = "Downstream_IDs",
        how = "left"
    )
    
    # If df also has Track_ID, join on both Track_ID and Source_ID to be safe:
    df_with_endpoints = df.merge(
         downstream_exploded,
         left_on = ["Track_ID", "Source_ID"],
         right_on = ["Track_ID", "Downstream_IDs"],
         how = "left"
     )
    
    # If multiple EndPoint_IDs match a given Source_ID, group them into a list
    df_with_endpoints = (
        df_with_endpoints.groupby(df.columns.tolist(), dropna = False)["EndPoint_ID"]
        .agg(lambda x: list(x.dropna()))
        .reset_index()
    )


    destination_splittingEvents_downstream = outdir + "_SplittingEventsDownstream.csv"
    df_with_endpoints.to_csv(destination_splittingEvents_downstream)
    print("Successfully exported endpoint dataframe to " + destination_splittingEvents_downstream)

    df = df.merge(df_with_endpoints)
    
    df["Position"] = position
    
    df["Dataset"] = dataset

    df["Condition"] = condition
  
    df["Generation"] = df["Lineage"].apply(lambda x: x.count(".") + 1)
    df["Mother_ID"] = df["Lineage"].apply(lambda x: get_mother(x))
    df["Grandmother_ID"] = df["Lineage"].apply(lambda x: get_grandmother(x))
    df["Sister_ID"] = df.apply(get_sister, dataframe = df, axis = 1)
    df["Daughters_ID"] = df.apply(get_daughters, dataframe = df, axis = 1)
    df["Number_of_daughters_in_mitosis"] = df['Daughters_ID'].str.len()
    df["Mother_Number_of_daughters_in_mitosis"] = df.apply(get_mother_n_of_daughters, dataframe = df, axis = 1)
    
    df["Cell_Cycle_mins"] = df.apply(get_cell_cycle, dataframe = df, axis = 1)
    df["Cell_Cycle_hours"] = df.Cell_Cycle_mins / 60

    # Randomising order of dataframe to 
    # avoid that sisters with shortest cell cycles
    # are 'False' for Seen_sister or seen_granny.
    df = df.sample(frac = 1) # shuffle rows
    
    # This will allow to sample one cell out of sister pair
    seen_sister_list = []
    print("Populating sister list")
    df["Seen_sister"] = df.apply(seensister, already_seen_list = seen_sister_list, axis = 1)    
    #df["Random_ID"] = df.apply(get_random, dataframe = df, axis = 1)
    
    # This will allow to sample one cell out of cousin quartett
    seen_granny_list = []
    print("Populating granny list")
    df["Seen_granny"] = df.apply(seengranny, granny_seen_list = seen_granny_list, axis = 1) 
    
    # sort for downstream analysis after shuffling
    df = df.sort_values(['Frame', 'Track_Coordinate_X'])
    destination = outdir + "_lineages.csv"
    df.to_csv(destination)
    print("Successfully exported lineage dataframe to " + destination)
    
    return df

In [16]:
df_02 = tracking_dataframes(
    edges_dir = root + "20250724/Position_2_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_edges.csv", 
    spots_dir = root + "20250724/Position_2_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_spots.csv",
    dataset = "20250724",
    position = 2,
    condition = "siLUC1",
    outdir = root + "20250724/Position_2_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03"
)

df_03 = tracking_dataframes(
    edges_dir = root + "20250724/Position_3_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_edges.csv", 
    spots_dir = root + "20250724/Position_3_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_spots.csv",
    dataset = "20250724",
    position = 3,
    condition = "siLUC1",
    outdir = root + "20250724/Position_3_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03"
)

df_04 = tracking_dataframes(
    edges_dir = root + "20250724/Position_4_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_edges.csv", 
    spots_dir = root + "20250724/Position_4_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_spots.csv",
    dataset = "20250724",
    position = 4,
    condition = "siLUC1",
    outdir = root + "20250724/Position_4_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03"
)

df_05 = tracking_dataframes(
    edges_dir = root + "20250724/Position_5_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_edges.csv", 
    spots_dir = root + "20250724/Position_5_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_spots.csv",
    dataset = "20250724",
    position = 5,
    condition = "siLUC1",
    outdir = root + "20250724/Position_5_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03"
)

df_06 = tracking_dataframes(
    edges_dir = root + "20250724/Position_6_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_edges.csv", 
    spots_dir = root + "20250724/Position_6_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_spots.csv",
    dataset = "20250724",
    position = 6,
    condition = "siHAUS6",
    outdir = root + "20250724/Position_6_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03"
)

df_07 = tracking_dataframes(
    edges_dir = root + "20250724/Position_7_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_edges.csv", 
    spots_dir = root + "20250724/Position_7_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_spots.csv",
    dataset = "20250724",
    position = 7,
    condition = "siHAUS6",
    outdir = root + "20250724/Position_7_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03"
)

df_08 = tracking_dataframes(
    edges_dir = root + "20250724/Position_8_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_edges.csv", 
    spots_dir = root + "20250724/Position_8_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_spots.csv",
    dataset = "20250724",
    position = 8,
    condition = "siHAUS6",
    outdir = root + "20250724/Position_8_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03"
)

df_09 = tracking_dataframes(
    edges_dir = root + "20250724/Position_9_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_edges.csv", 
    spots_dir = root + "20250724/Position_9_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_spots.csv",
    dataset = "20250724",
    position = 9,
    condition = "siHAUS6",
    outdir = root + "20250724/Position_9_siHAUS6/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03"
)

####

df_10 = tracking_dataframes(
    edges_dir = root + "20250807/Position_2_siLUC1/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_edges.csv", 
    spots_dir = root + "20250807/Position_2_siLUC1/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_spots.csv",
    dataset = "20250807",
    position = 2,
    condition = "siLUC1",
    outdir = root + "20250807/Position_2_siLUC1/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04"
)

df_11 = tracking_dataframes(
    edges_dir = root + "20250807/Position_3_siLUC1/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_edges.csv", 
    spots_dir = root + "20250807/Position_3_siLUC1/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_spots.csv",
    dataset = "20250807",
    position = 3,
    condition = "siLUC1",
    outdir = root + "20250807/Position_3_siLUC1/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04"
)

df_12 = tracking_dataframes(
    edges_dir = root + "20250807/Position_4_siLUC1/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_edges.csv", 
    spots_dir = root + "20250807/Position_4_siLUC1/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_spots.csv",
    dataset = "20250807",
    position = 4,
    condition = "siLUC1",
    outdir = root + "20250807/Position_4_siLUC1/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04"
)

df_13 = tracking_dataframes(
    edges_dir = root + "20250807/Position_5_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_edges.csv", 
    spots_dir = root + "20250807/Position_5_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_spots.csv",
    dataset = "20250807",
    position = 5,
    condition = "siHAUS6",
    outdir = root + "20250807/Position_5_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04"
)

df_14 = tracking_dataframes(
    edges_dir = root + "20250807/Position_6_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_edges.csv", 
    spots_dir = root + "20250807/Position_6_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_spots.csv",
    dataset = "20250807",
    position = 6,
    condition = "siHAUS6",
    outdir = root + "20250807/Position_6_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04"
)

df_15 = tracking_dataframes(
    edges_dir = root + "20250807/Position_7_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_edges.csv", 
    spots_dir = root + "20250807/Position_7_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_spots.csv",
    dataset = "20250807",
    position = 7,
    condition = "siHAUS6",
    outdir = root + "20250807/Position_7_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04"
)

df_16 = tracking_dataframes(
    edges_dir = root + "20250807/Position_8_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_edges.csv", 
    spots_dir = root + "20250807/Position_8_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_spots.csv",
    dataset = "20250807",
    position = 8,
    condition = "siHAUS6",
    outdir = root + "20250807/Position_8_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04"
)

df_17 = tracking_dataframes(
    edges_dir = root + "/20250807/Position_9_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_edges.csv", 
    spots_dir = root + "/20250807/Position_9_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04_spots.csv",
    dataset = "20250807",
    position = 9,
    condition = "siHAUS6",
    outdir = root + "/20250807/Position_9_siHAUS6/Concatenated/20250807_IM_H2B-GFP_MitoStop_RNAi04"
)

####

df_18 = tracking_dataframes(
    edges_dir = root + "/20250814/Position_2_siLUC1/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_edges.csv", 
    spots_dir = root + "/20250814/Position_2_siLUC1/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_spots.csv",
    dataset = "20250814",
    position = 2,
    condition = "siLUC1",
    outdir = root + "/20250814/Position_2_siLUC1/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05"
)

df_19 = tracking_dataframes(
    edges_dir = root + "/20250814/Position_3_siLUC1/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_edges.csv", 
    spots_dir = root + "/20250814/Position_3_siLUC1/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_spots.csv",
    dataset = "20250814",
    position = 3,
    condition = "siLUC1",
    outdir = root + "/20250814/Position_3_siLUC1/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05"
)

df_20 = tracking_dataframes(
    edges_dir = root + "/20250814/Position_4_siLUC1/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_edges.csv", 
    spots_dir = root + "/20250814/Position_4_siLUC1/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_spots.csv",
    dataset = "20250814",
    position = 4,
    condition = "siLUC1",
    outdir = root + "/20250814/Position_4_siLUC1/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05"
)

df_21 = tracking_dataframes(
    edges_dir = root + "/20250814/Position_5_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_edges.csv", 
    spots_dir = root + "/20250814/Position_5_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_spots.csv",
    dataset = "20250814",
    position = 5,
    condition = "siHAUS6",
    outdir = root + "/20250814/Position_5_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05"
)

df_22 = tracking_dataframes(
    edges_dir = root + "/20250814/Position_6_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_edges.csv", 
    spots_dir = root + "/20250814/Position_6_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_spots.csv",
    dataset = "20250814",
    position = 6,
    condition = "siHAUS6",
    outdir = root + "/20250814/Position_6_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05"
)

df_23 = tracking_dataframes(
    edges_dir = root + "/20250814/Position_7_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_edges.csv", 
    spots_dir = root + "/20250814/Position_7_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_spots.csv",
    dataset = "20250814",
    position = 7,
    condition = "siHAUS6",
    outdir = root + "/20250814/Position_7_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05"
)

df_24 = tracking_dataframes(
    edges_dir = root + "/20250814/Position_8_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_edges.csv", 
    spots_dir = root + "/20250814/Position_8_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_spots.csv",
    dataset = "20250814",
    position = 8,
    condition = "siHAUS6",
    outdir = root + "/20250814/Position_8_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05"
)

# below not used because FOV is too confluent

#df_25 = tracking_dataframes(
#    edges_dir = root + "/20250814/Position_9_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_edges.csv", 
#    spots_dir = root + "/20250814/Position_9_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05_spots.csv",
#    dataset = "20250814",
#    position = 9,
#    outdir = root + "/20250814/Position_9_siHAUS6/Concatenated/20250814_IM_H2B-GFP_MitoStop_RNAi05"
#)

####### Integrating second generation datasets ########

df_25 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_01_siLUC1/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_01_siLUC1/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 1,
    condition = "siLUC1",
    outdir = root + "/20251204/Position_01_siLUC1/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

df_26 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_02_siLUC1/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_02_siLUC1/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 2,
    condition = "siLUC1",
    outdir = root + "/20251204/Position_02_siLUC1/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

df_27 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_03_siLUC1/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_03_siLUC1/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 3,
    condition = "siLUC1",
    outdir = root + "/20251204/Position_03_siLUC1/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

df_28 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_04_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_04_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 4,
    condition = "siHAUS6",
    outdir = root + "/20251204/Position_04_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

df_29 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_05_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_05_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 5,
    condition = "siHAUS6",
    outdir = root + "/20251204/Position_05_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

df_30 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_06_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_06_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 6,
    condition = "siHAUS6",
    outdir = root + "/20251204/Position_06_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

df_31 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_06_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_06_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 6,
    condition = "siHAUS6",
    outdir = root + "/20251204/Position_06_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

df_32 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_07_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_07_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 7,
    condition = "siHAUS6",
    outdir = root + "/20251204/Position_07_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

df_33 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_08_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_08_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 8,
    condition = "siHAUS6",
    outdir = root + "/20251204/Position_08_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

df_34 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_09_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_09_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 9,
    condition = "siHAUS6",
    outdir = root + "/20251204/Position_09_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

df_35 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_10_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_10_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 10,
    condition = "siHAUS6",
    outdir = root + "/20251204/Position_10_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

df_36 = tracking_dataframes(
    edges_dir = root + "/20251204/Position_11_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_edges.csv", 
    spots_dir = root + "/20251204/Position_11_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201_spots.csv",
    dataset = "20251204",
    position = 11,
    condition = "siHAUS6",
    outdir = root + "/20251204/Position_11_siHAUS6/Max/20251204_IM_H2B-GFP_MitoStop_RNAi201"
)

######
# Augmin depletion experiment 202 (FBS 20%)
# For comparison, crop tracking tables to have the same frame numbers

df_37 = tracking_dataframes(
    edges_dir = root + "/20251215/Position_01_Mock/Max/20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "/20251215/Position_01_Mock/Max/20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 1,
    condition = "siMock",
    outdir = root + "/20251215/Position_01_Mock/Max/20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_38 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_02_Mock\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_02_Mock\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 2,
    condition = "siMock",
    outdir = root + "\\20251215\\Position_02_Mock\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_39 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_03_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_03_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 3,
    condition = "siLUC1",
    outdir = root + "\\20251215\\Position_03_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_40 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_04_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_04_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 4,
    condition = "siLUC1",
    outdir = root + "\\20251215\\Position_04_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202" 
)

df_41 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_05_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_05_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 5,
    condition = "siLUC1",
    outdir = root + "\\20251215\\Position_05_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_42 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_06_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_06_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 6,
    condition = "siLUC1",
    outdir = root + "\\20251215\\Position_06_siLUC1\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_43 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_07_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_07_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 7,
    condition = "siHAUS6",
    outdir = root + "\\20251215\\Position_07_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_44 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_08_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_08_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 8,
    condition = "siHAUS6",
    outdir = root + "\\20251215\\Position_08_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_45 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_09_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_09_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 9,
    condition = "siHAUS6",
    outdir = root + "\\20251215\\Position_09_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_46 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_10_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_10_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 10,
    condition = "siHAUS6",
    outdir = root + "\\20251215\\Position_10_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_47 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_11_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_11_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 11,
    condition = "siHAUS6",
    outdir = root + "\\20251215\\Position_11_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_48 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_12_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_12_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 12,
    condition = "siHAUS6",
    outdir = root + "\\20251215\\Position_12_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_49 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_13_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_13_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 13,
    condition = "siHAUS6",
    outdir = root + "\\20251215\\Position_13_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

df_50 = tracking_dataframes(
    edges_dir = root + "\\20251215\\Position_14_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_edges.csv", 
    spots_dir = root + "\\20251215\\Position_14_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202_spots.csv",
    dataset = "20251215",
    position = 14,
    condition = "siHAUS6",
    outdir = root + "\\20251215\\Position_14_siHAUS6\\Max\\20251215_IM_H2B-GFP_MitoStop_RNAi202"
)

#####################
#USP28 double depletion experiments 01
#####################

df_51 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_01_siLUC1\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_01_siLUC1\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 1,
    condition = "siLUC1",
    outdir = root + "\\20260126_USP28\\Position_01_siLUC1\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_52 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_02_siLUC1\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_02_siLUC1\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 2,
    condition = "siLUC1",
    outdir = root + "\\20260126_USP28\\Position_02_siLUC1\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_53 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_03_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_03_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 3,
    condition = "siLUC1siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_03_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_54 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_04_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_04_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 4,
    condition = "siLUC1siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_04_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_55 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_05_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_05_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 5,
    condition = "siLUC1siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_05_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_56 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_06_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_06_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 6,
    condition = "siLUC1siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_06_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_57 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_07_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_07_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 7,
    condition = "siLUC1siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_07_siLUC1siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_58 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_08_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_08_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 8,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_08_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_59 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_09_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_09_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 9,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_09_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_60 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_10_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_10_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 10,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_10_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_61 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_11_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_11_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 11,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_11_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_62 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_12_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_12_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 12,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_12_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_63 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_13_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_13_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 13,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_13_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

df_64 = tracking_dataframes(
    edges_dir = root + "\\20260126_USP28\\Position_14_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_edges.csv", 
    spots_dir = root + "\\20260126_USP28\\Position_14_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_spots.csv",
    dataset = "20260126",
    position = 14,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260126_USP28\\Position_14_siUSP28siHAUS6\\Max\\20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01"
)

#####################
#USP28 double depletion experiments 02
#####################

df_65 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_01_siLUC\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_01_siLUC\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 1,
    condition = "siLUC1",
    outdir = root + "\\20260205_USP28\\Position_01_siLUC\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_66 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_02_siLUC\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_02_siLUC\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 2,
    condition = "siLUC1",
    outdir = root + "\\20260205_USP28\\Position_02_siLUC\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_67 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_03_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_03_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 3,
    condition = "siLUC1siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_03_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_68 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_04_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_04_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 4,
    condition = "siLUC1siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_04_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_69 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_05_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_05_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 5,
    condition = "siLUC1siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_05_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_70 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_06_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_06_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 6,
    condition = "siLUC1siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_06_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_71 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_07_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_07_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 7,
    condition = "siLUC1siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_07_siLUCsiHAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_72 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_08_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_08_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 8,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_08_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_73 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_09_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_09_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 9,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_09_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_74 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_10_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_10_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 10,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_10_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_75 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_11_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_11_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 11,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_11_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_76 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_12_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_12_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 12,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_12_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_77 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_13_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_13_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 13,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_13_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

df_78 = tracking_dataframes(
    edges_dir = root + "\\20260205_USP28\\Position_14_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_edges.csv", 
    spots_dir = root + "\\20260205_USP28\\Position_14_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_spots.csv",
    dataset = "20260205",
    position = 14,
    condition = "siUSP28siHAUS6",
    outdir = root + "\\20260205_USP28\\Position_14_siUSP28HAUS6\\Max\\20260205_IM_H2B-GFP_MitoStop_RNAiUSP28"
)

Successfully exported endpoint dataframe to Y:/_Tobias/LSM980/Mitotic_Stopwatch/20250724/Position_2_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream.csv
Successfully annotated lineage information.
Successfully exported endpoint dataframe to Y:/_Tobias/LSM980/Mitotic_Stopwatch/20250724/Position_2_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_SplittingEventsDownstream.csv
Populating sister list
Populating granny list
Successfully exported lineage dataframe to Y:/_Tobias/LSM980/Mitotic_Stopwatch/20250724/Position_2_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_lineages.csv
Successfully exported endpoint dataframe to Y:/_Tobias/LSM980/Mitotic_Stopwatch/20250724/Position_3_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream.csv
Successfully annotated lineage information.
Successfully exported endpoint dataframe to Y:/_Tobias/LSM980/Mitotic_Stopwatch/20250724/Position_3_siLUC1/Concatenated/20250724_IM_H2B-GFP_MitoStop_RNAi03_SplittingEvent

In [17]:
# Collect, concatenate and save dataframe

def collect_dataframes(start, end, prefix = "df_"):
    dfs = {}
    for i in range(start, end + 1):
        name_padded = f"{prefix}{i:02d}"
        name_plain = f"{prefix}{i}"

        if name_padded in globals():
            dfs[i] = globals()[name_padded]
        elif name_plain in globals():
            dfs[i] = globals()[name_plain]

    return dfs
dataframes = collect_dataframes(2, 78) # Specify dataframes to include
print(sorted(dataframes.keys()))

df_concat = pd.concat(dataframes.values(), ignore_index = True)
df_concat.to_csv(outputFolder + "MainDataFrame_Lineages_Augmin.csv")

[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78]


In [18]:
# Summary of lineage statistics

from statistics import mean

max_frame = 823 # this crops the max time that is included in the statistics 
# TODO in the next script, cut the divisions that are included for the generation analysis

def get_lineage_statistics(max_frame = max_frame, dataframe = dataframes, outDir = outputFolder):
    
    dataframes_for_concat = []
        
    for data in dataframe.values():
        data["Frame"] = data.Frame.astype(int)

        data = data[data["Frame"] < max_frame]
        

        first_row = data.head(1)
        dataset = first_row["Dataset"].iloc[0]
        position = first_row["Position"].iloc[0]
        condition = first_row["Condition"].iloc[0]

        max_generation = data.Generation.max()
        No_of_source_ids = data.Source_ID.dropna().shape[0]
        #No_of_random_ids = data.Random_ID.dropna().shape[0]
        No_of_sister_ids = data.Sister_ID.dropna().shape[0]
        No_of_mother_ids = data.Mother_ID.dropna().shape[0]
        #No_of_aunt_ids = data.Aunt_ID.dropna().shape[0]
        #No_of_cousin_ids = data.Cousin_ID.dropna().shape[0]
        No_of_grandmother_ids = data.Grandmother_ID.dropna().shape[0]
        
        list_of_tracks = data.Track_ID.unique()
        list_of_trackSubData = []
        for track in list_of_tracks:
            sub_data = data.loc[data.Track_ID == track]
            list_of_trackSubData.append(sub_data)

        No_of_tracks = data.Track_ID.nunique()
        Average_No_of_splitting_events = data.groupby("Track_ID").describe().count().mean()

        sister_per_split = []
        mother_per_split = []
        grandmother_per_split = []

        for sub_data in list_of_trackSubData: 
            number_of_sister_ids = sub_data.Sister_ID.dropna().shape[0]
            sis_per_total_splits = number_of_sister_ids / sub_data.shape[0]
            sister_per_split.append(sis_per_total_splits)

            number_of_mother_ids = sub_data.Mother_ID.dropna().shape[0]
            mom_per_total_splits = number_of_mother_ids / sub_data.shape[0]
            mother_per_split.append(mom_per_total_splits)

            number_of_gmother_ids = sub_data.Grandmother_ID.dropna().shape[0]
            gmom_per_total_splits = number_of_gmother_ids / sub_data.shape[0]
            grandmother_per_split.append(gmom_per_total_splits)

        statistics_dict = {"Dataset": int(dataset), 
                           "Position": position,
                           "Condition": condition,
                           "Number_of_tracks": No_of_tracks,
                           "Max_No_of_Generations": max_generation,
                           "Number_of_Source_IDs": No_of_source_ids,
                           "Number_of_Sister_IDs": No_of_sister_ids,
                           "Number_of_Mother_IDs": No_of_mother_ids,
                           "Number_of_Grandmother_IDs": No_of_grandmother_ids,
                           #"Number_of_Random_IDs": No_of_random_ids,
                           "Average_No_of_SplittingEvents_per_Track": data.shape[0] / No_of_tracks, 
                           "Average_No_of_Sisters_per_Splitting_event": mean(sister_per_split), 
                           "Average_No_of_Mothers_per_Splitting_event": mean(mother_per_split), 
                           "Average_No_of_Grandmothers_per_Splitting_event": mean(grandmother_per_split), 
                          }
        df = pd.DataFrame(statistics_dict, index = [0])
        dataframes_for_concat.append(df)

    final_df = pd.concat(dataframes_for_concat)
    
    final_df.to_csv(outDir + "MetaStatistics_TrackingLineages_Augmin.csv")
    return final_df

statistics_df = get_lineage_statistics()
statistics_df

,Dataset,Position,Condition,Number_of_tracks,Max_No_of_Generations,Number_of_Source_IDs,Number_of_Sister_IDs,Number_of_Mother_IDs,Number_of_Grandmother_IDs,Average_No_of_SplittingEvents_per_Track,Average_No_of_Sisters_per_Splitting_event,Average_No_of_Mothers_per_Splitting_event,Average_No_of_Grandmothers_per_Splitting_event
0,20250724,2,siLUC1,18,3,48,20,30,11,2.666667,0.236508,0.434656,0.109524
0,20250724,3,siLUC1,14,3,38,14,23,9,2.714286,0.192177,0.412415,0.118197
0,20250724,4,siLUC1,44,4,102,42,58,19,2.318182,0.225866,0.351677,0.089123
0,20250724,5,siLUC1,36,4,91,40,55,23,2.527778,0.243519,0.378858,0.122685
0,20250724,6,siHAUS6,46,3,91,36,44,5,1.978261,0.230228,0.310663,0.016770
...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,20260205,10,siUSP28siHAUS6,16,3,54,32,38,11,3.375000,0.514583,0.644792,0.142708
0,20260205,11,siUSP28siHAUS6,16,3,50,28,34,8,3.125000,0.472917,0.623958,0.111458
0,20260205,12,siUSP28siHAUS6,17,3,67,40,50,22,3.941176,0.479272,0.690056,0.231933
0,20260205,13,siUSP28siHAUS6,13,3,42,26,29,6,3.230769,0.543590,0.658974,0.092308


In [19]:
# Quick visualisation

import altair as alt

alt.Chart(
    statistics_df
).mark_bar().encode(
    x = 'Condition',
    y = 'mean(Average_No_of_SplittingEvents_per_Track)',
    color = 'Condition',
    column = 'Dataset'
)

alt.Chart(...)

In [20]:
alt.Chart(
    df_concat
).mark_bar().encode(
    x='Condition:O',
    y='mean(Cell_Cycle_hours)',
    column='Dataset'
)

alt.Chart(...)